In [1]:
import boto3
import sagemaker

from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.workflow.model_step import ModelStep
from sagemaker.workflow.step_collections import RegisterModel
from sagemaker.workflow.properties import PropertyFile
from sagemaker.workflow.pipeline_context import PipelineSession


from sagemaker.processing import (
    ScriptProcessor,
    ProcessingInput,
    ProcessingOutput,
)
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
from sagemaker.model_metrics import ModelMetrics, MetricsSource

pipeline_session = PipelineSession()

session = sagemaker.Session()
region = boto3.Session().region_name
role = sagemaker.get_execution_role()

bucket = session.default_bucket()
prefix = "sagemaker"

print("Region :", region)
print("Bucket :", bucket)
print("Role   :", role)
print("SageMaker SDK:", sagemaker.__version__)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Region : us-east-1
Bucket : sagemaker-us-east-1-140083316867
Role   : arn:aws:iam::140083316867:role/service-role/AmazonSageMaker-ExecutionRole-20260429T143977
SageMaker SDK: 2.245.0


In [2]:
xgb_image = sagemaker.image_uris.retrieve(
    framework="xgboost",
    region=region,
    version="1.7-1",          # ← 明示する
    image_scope="training",
)


processor = ScriptProcessor(
    image_uri=xgb_image,
    role=role,
    instance_type="ml.m5.large",
    instance_count=1,
    command=["python3"],
    sagemaker_session=pipeline_session,
)


INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


In [3]:
step_preprocess = ProcessingStep(
    name="Preprocess",
    processor=processor,
    inputs=[
        ProcessingInput(
            source=f"s3://{bucket}/sagemaker/raw/abalone.csv",
            destination="/opt/ml/processing/input"
        )
    ],
    outputs=[
        ProcessingOutput(
            source="/opt/ml/processing/train",
            output_name="train"
        ),
        ProcessingOutput(
            source="/opt/ml/processing/validation",
            output_name="validation"
        )
    ],
    code="preprocess.py",
)

In [4]:
estimator = Estimator(
    image_uri=xgb_image,
    role=role,
    instance_type="ml.m5.large",
    instance_count=1,
    output_path=f"s3://{bucket}/{prefix}/training",
    sagemaker_session=pipeline_session,
)

estimator.set_hyperparameters(
    objective="reg:squarederror",
    eval_metric="rmse",
    num_round=200,
    max_depth=5,
    eta=0.2,
    subsample=0.8,
    colsample_bytree=0.8,
)

step_train = TrainingStep(
    name="Train",
    estimator=estimator,
    inputs={
        "train": TrainingInput(
            s3_data=step_preprocess.properties
            .ProcessingOutputConfig
            .Outputs["train"]
            .S3Output
            .S3Uri,
            content_type="text/csv"
        ),
        "validation": TrainingInput(
            s3_data=step_preprocess.properties
            .ProcessingOutputConfig
            .Outputs["validation"]
            .S3Output
            .S3Uri,
            content_type="text/csv"
        ),
    },
)

In [5]:
evaluation_report = PropertyFile(
    name="EvaluationReport",
    output_name="evaluation",
    path="metrics.json",
)

step_evaluate = ProcessingStep(
    name="Evaluate",
    processor=processor,
    inputs=[
        ProcessingInput(
            source=step_train.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/model",
        ),
        ProcessingInput(
            source=step_preprocess.properties
            .ProcessingOutputConfig
            .Outputs["validation"]
            .S3Output
            .S3Uri,
            destination="/opt/ml/processing/validation",
        ),
    ],
    outputs=[
        ProcessingOutput(
            source="/opt/ml/processing/evaluation",
            output_name="evaluation"
        )
    ],
    code="evaluate.py",
    property_files=[evaluation_report],
)

In [6]:
from sagemaker.workflow.functions import Join
model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri=Join(
            on="",
            values=[
                step_evaluate.properties
                .ProcessingOutputConfig
                .Outputs["evaluation"]
                .S3Output
                .S3Uri,
                "/metrics.json",
            ],
        ),
        content_type="application/json",
    )
)


In [7]:
step_register = RegisterModel(
    name="RegisterModel",
    estimator=estimator,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    content_types=["text/csv"],
    response_types=["text/csv"],
    inference_instances=["ml.m5.large"],
    transform_instances=["ml.m5.large"],
    model_package_group_name="abalone-xgboost",
    model_metrics=model_metrics,
    approval_status="PendingManualApproval",
)


In [8]:
pipeline = Pipeline(
    name="abalone-pipeline",
    steps=[
        step_preprocess,
        step_train,
        step_evaluate,
        step_register,
    ],
    sagemaker_session=pipeline_session,
)


In [9]:
pipeline.upsert(role_arn=role)
pipeline.definition()

'{"Version": "2020-12-01", "Metadata": {}, "Parameters": [], "PipelineExperimentConfig": {"ExperimentName": {"Get": "Execution.PipelineName"}, "TrialName": {"Get": "Execution.PipelineExecutionId"}}, "Steps": [{"Name": "Preprocess", "Type": "Processing", "Arguments": {"ProcessingResources": {"ClusterConfig": {"InstanceType": "ml.m5.large", "InstanceCount": 1, "VolumeSizeInGB": 30}}, "AppSpecification": {"ImageUri": "683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1", "ContainerEntrypoint": ["python3", "/opt/ml/processing/input/code/preprocess.py"]}, "RoleArn": "arn:aws:iam::140083316867:role/service-role/AmazonSageMaker-ExecutionRole-20260429T143977", "ProcessingInputs": [{"InputName": "input-1", "AppManaged": false, "S3Input": {"S3Uri": "s3://sagemaker-us-east-1-140083316867/sagemaker/raw/abalone.csv", "LocalPath": "/opt/ml/processing/input", "S3DataType": "S3Prefix", "S3InputMode": "File", "S3DataDistributionType": "FullyReplicated", "S3CompressionType": "None"}}, {

In [10]:
execution = pipeline.start()
execution.wait()